In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from timm.layers.cbam import SpatialAttn
from torchvision.models.mobilenetv3 import mobilenet_v3_large, MobileNet_V3_Large_Weights, InvertedResidual
from torchvision.ops.misc import Conv2dNormActivation
import torchprofile
from PIL import Image
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from imblearn.metrics import sensitivity_score, specificity_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from functools import partial
import time
import sys
import os

c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(sys.executable)

c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Scripts\python.exe


In [206]:
%load_ext tensorboard
%tensorboard --logdir runs/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 23484), started 1 day, 23:00:59 ago. (Use '!kill 23484' to kill it.)

In [207]:
try:
  from google.colab import drive
  drive.mount('drive')
  data_path = "drive/MyDrive/Colab Notebooks/Datasets"
except ImportError:
  if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = f"kaggle/input"
  else:
    data_path = "../data"

In [208]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

**Enable cuda if available**

In [ ]:
cuda_id = 0
device = torch.device(f'cuda:{cuda_id}' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA Version: 12.9


**Define dataset pipeline**

In [210]:
class ISIC2019(Dataset): # TODO: consider maybe removing downsampled or removing duplicates. unsure if these are necessary
    def __init__(self, annotations_file, img_dir, split, transform=None, target_transform=None):
        img_labels = pd.read_csv(annotations_file)

        train_samples = img_labels.sample(frac=0.80, random_state=42) # randomstate for consistency across objects
        valid_samples = img_labels.drop(train_samples.index)

        if split == 'train':
            self.img_labels = train_samples

        if split == 'valid':
            self.img_labels = valid_samples

        self.img_labels.drop("UNK", axis=1, inplace=True) # remove unknown category
        self.img_labels.reset_index(drop=True, inplace=True) # reset index for stability
        ohe_labels = self.img_labels.iloc[:, 1:]

        self.labels = np.where(ohe_labels==1)[1]

        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform
        
        self.class_names = ohe_labels.columns.to_list()
        self.num_classes = len(self.class_names)

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.img_labels.iloc[idx, 0]}.jpg")
        image = Image.open(img_path)
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            label = self.target_transform(label)

        return image, label

**MobileNet Architecture**

In [ ]:
class MobileNetV3_Baseline(nn.Module):
    def __init__(self, load_imagenet_weights=True, freeze_net=True):
        super().__init__()
        self.num_classes = 8
        self.mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT if load_imagenet_weights else None)
        
        if freeze_net:
            for param in self.mobilenet.parameters():
                param.requires_grad = False
        # freeze mobilenet layers if enabled

        in_features = self.mobilenet.classifier[3].in_features
        self.mobilenet.classifier[3] = nn.Linear(in_features, self.num_classes) 
        # replace last layer with custom number of output categories

        nn.init.normal_(self.mobilenet.classifier[3].weight, 0, 0.01) 
        nn.init.zeros_(self.mobilenet.classifier[3].bias) 
        # use same weight + bias init as pytorch for consistency

    def forward(self, x):
        x = self.mobilenet(x)
        return x

**Hyperparameters**

In [ ]:
epochs = 100
learning_rate = 1e-4
batch_size = 16
early_stopping_rounds = 100
start_epoch = 0

freeze_net = False
load_imagenet_weights = True

**Move model to specified device**

In [ ]:
model = MobileNetV3_Baseline(load_imagenet_weights=load_imagenet_weights, freeze_net=freeze_net).to(device=device)
model_name = model.__class__.__name__

In [217]:
if start_epoch > 0:
    model.load_state_dict(torch.load(f"checkpoints/{model_name}/epoch-{start_epoch}.pth")) # load specific checkpoint

output_dir = f"checkpoints/{model_name}"
os.makedirs(output_dir, exist_ok=True) # create baseline dir for checkpoints if not exist

**Summary of model**

In [218]:
summary(model, input_size=(batch_size, 3, 224, 224))

Layer (type:depth-idx)                                            Output Shape              Param #
MobileNetV3_Baseline                                              [16, 8]                   --
├─MSF: 1-1                                                        [16, 3, 224, 224]         --
│    └─ModuleList: 2-1                                            --                        --
│    │    └─Conv2dNormActivation: 3-1                             [16, 1, 224, 224]         5
│    │    └─Conv2dNormActivation: 3-2                             [16, 1, 224, 224]         29
│    │    └─Conv2dNormActivation: 3-3                             [16, 1, 224, 224]         77
│    │    └─Conv2dNormActivation: 3-4                             [16, 1, 224, 224]         149
│    └─Conv2dNormActivation: 2-2                                  [16, 3, 224, 224]         --
│    │    └─Conv2d: 3-5                                           [16, 3, 224, 224]         12
│    │    └─BatchNorm2d: 3-6                 

**FLOPS**

In [219]:
macs = torchprofile.profile_macs(model, torch.randn(1, 3, 224, 224).to(device))
macs

c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Lib\site-packages\torchprofile\profile.py:22: UserWarning: No handlers found: "aten::hardswish_". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Lib\site-packages\torchprofile\profile.py:22: UserWarning: No handlers found: "aten::permute". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Lib\site-packages\torchprofile\profile.py:22: UserWarning: No handlers found: "aten::split_with_sizes". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
c:\Users\abhin\Documents\source\Skin Lesion Classification\env\Lib\site-packages\torchprofile\profile.py:22: UserWarning: No handlers found: "aten::amax". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(


260562924

**Transform Pipelines**

In [220]:
train_transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

**Dataset + Loss + Optimizer**

In [221]:
train_dataset = ISIC2019(img_dir=f'{data_path}/ISIC_2019_Training_Input', annotations_file=f'{data_path}/ISIC_2019_Training_GroundTruth.csv', split='train', transform=train_transform)
valid_dataset = ISIC2019(img_dir=f'{data_path}/ISIC_2019_Training_Input', annotations_file=f'{data_path}/ISIC_2019_Training_GroundTruth.csv', split='valid', transform=valid_transform)

loss = nn.CrossEntropyLoss()
unfrozen_params = filter(lambda p: p.requires_grad, model.parameters()) # only keep params which are unfrozen for optimizer
optimizer = torch.optim.Adam(unfrozen_params, lr=learning_rate)

**SummaryWriter**

In [ ]:
timestamp = time.strftime("%b-%d-%Y %I-%M-%S %p")
writer = SummaryWriter(log_dir=f"runs/{timestamp} {model_name} lr_{learning_rate} {'pretrained' if load_imagenet_weights else ''} {'frozen_net' if freeze_net else ''} batch_size_{batch_size}", flush_secs=30)

**DataLoader**

In [223]:
dataloader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
dataloader_valid = DataLoader(valid_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

**Core Loop**

In [ ]:
epochs_without_gain = 0
best_valid_loss = float("inf")

max_train_acc = 0
max_valid_acc = 0

num_batches_trained = 0
num_samples_trained = 0

print("Beginning training")
for epoch in range(start_epoch, epochs):
    train_epoch_loss, valid_epoch_loss = 0.0, 0.0

    train_epoch_preds, train_epoch_labels, train_epoch_probs = [], [], []
    valid_epoch_preds, valid_epoch_labels, valid_epoch_probs = [], [], []

    train_start_time = time.time()
    model.train()
    for batch_idx, (train_features, train_labels) in enumerate(dataloader_train):
        train_batch_size = len(train_labels)
        train_features = train_features.to(device)
        train_labels = train_labels.to(device) # move to device
        
        optimizer.zero_grad()

        predictions = model(train_features)
        probs = torch.softmax(predictions, dim=1)
        predictions_labels = torch.argmax(predictions, dim=1)

        train_epoch_preds.extend(predictions_labels.cpu().numpy())
        train_epoch_probs.extend(probs.detach().cpu().numpy())
        train_epoch_labels.extend(train_labels.cpu().numpy())

        train_batch_loss = loss(predictions, train_labels)
        train_batch_loss.backward()

        optimizer.step()

        train_epoch_loss += train_batch_loss.item() * train_batch_size

        num_batches_trained += 1
        num_samples_trained += train_batch_size

        train_batch_acc = accuracy_score(train_labels.cpu().numpy(), predictions_labels.cpu().numpy())

        model.eval()
        with torch.inference_mode():
            valid_features, valid_labels = next(iter(dataloader_valid)) # sample random validation mini batch
            valid_features = valid_features.to(device)
            valid_labels = valid_labels.to(device) # move to device

            predictions = model(valid_features)
            predictions_labels = torch.argmax(predictions, dim=1)

            valid_batch_loss = loss(predictions, valid_labels)
            valid_batch_acc = accuracy_score(valid_labels.cpu().numpy(), predictions_labels.cpu().numpy())
        model.train()

        writer.add_scalar("Loss/train-batch", train_batch_loss.item(), num_batches_trained)
        writer.add_scalar("Accuracy/train-batch", train_batch_acc, num_batches_trained)

        writer.add_scalar("Loss/valid-batch", valid_batch_loss.item(), num_batches_trained)
        writer.add_scalar("Accuracy/valid-batch", valid_batch_acc, num_batches_trained)
    train_end_time = time.time()

    valid_start_time = time.time()
    model.eval()
    with torch.inference_mode():
        for batch_idx, (valid_features, valid_labels) in enumerate(dataloader_valid):
            valid_batch_size = len(valid_labels)
            valid_features = valid_features.to(device)
            valid_labels = valid_labels.to(device) # move to device
            
            predictions = model(valid_features)
            probs = torch.softmax(predictions, dim=1)
            predictions_labels = torch.argmax(predictions, dim=1)

            valid_epoch_preds.extend(predictions_labels.cpu().numpy())
            valid_epoch_probs.extend(probs.detach().cpu().numpy())
            valid_epoch_labels.extend(valid_labels.cpu().numpy())

            valid_batch_loss = loss(predictions, valid_labels)
            valid_epoch_loss += valid_batch_loss.item() * valid_batch_size
    valid_end_time = time.time()

    train_epoch_loss /= len(train_dataset)
    valid_epoch_loss /= len(valid_dataset)

    # TODO - check if roc auc score is computed correctly.

    y_train_score = np.vstack(train_epoch_probs)
    y_train_onehot = label_binarize(train_epoch_labels, classes=np.arange(model.num_classes))

    y_valid_score = np.vstack(valid_epoch_probs)
    y_valid_onehot = label_binarize(valid_epoch_labels, classes=np.arange(model.num_classes))

    train_epoch_acc = accuracy_score(train_epoch_labels, train_epoch_preds)
    train_epoch_prec = precision_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_rec = recall_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_f1 = f1_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_auc = roc_auc_score(y_train_onehot, y_train_score, average='macro', multi_class='ovr')
    train_epoch_spec = specificity_score(train_epoch_labels, train_epoch_preds, average='macro')

    valid_epoch_acc = accuracy_score(valid_epoch_labels, valid_epoch_preds)
    valid_epoch_prec = precision_score(valid_epoch_labels, valid_epoch_preds, average='macro')
    valid_epoch_rec = recall_score(valid_epoch_labels, valid_epoch_preds, average='macro')
    valid_epoch_f1 = f1_score(valid_epoch_labels, valid_epoch_preds, average='macro')
    valid_epoch_auc = roc_auc_score(y_valid_onehot, y_valid_score, average='macro', multi_class='ovr')
    valid_epoch_spec = specificity_score(valid_epoch_labels, valid_epoch_preds, average='macro')

    max_train_acc = max(max_train_acc, train_epoch_acc)
    max_valid_acc = max(max_valid_acc, valid_epoch_acc)

    writer.add_scalar("Loss/train-epoch", train_epoch_loss, epoch)
    writer.add_scalar('Accuracy/train-epoch', train_epoch_acc, epoch)
    writer.add_scalar('F1-Score/train-epoch', train_epoch_f1, epoch)
    writer.add_scalar('Precision/train-epoch', train_epoch_prec, epoch)
    writer.add_scalar('Recall-Sensitivity/train-epoch', train_epoch_rec, epoch)
    writer.add_scalar('AUC/train-epoch', train_epoch_auc, epoch)
    writer.add_scalar('Specificity/train-epoch', train_epoch_spec, epoch)
    writer.add_scalar('Elapsed Time/train-epoch-secs', train_end_time - train_start_time, epoch)

    writer.add_scalar("Loss/valid-epoch", valid_epoch_loss, epoch)
    writer.add_scalar('Accuracy/valid-epoch', valid_epoch_acc, epoch)
    writer.add_scalar('F1-Score/valid-epoch', valid_epoch_f1, epoch)
    writer.add_scalar('Precision/valid-epoch', valid_epoch_prec, epoch)
    writer.add_scalar('Recall-Sensitivity/valid-epoch', valid_epoch_rec, epoch)
    writer.add_scalar('AUC/valid-epoch', valid_epoch_auc, epoch)
    writer.add_scalar('Specificity/valid-epoch', valid_epoch_spec, epoch)
    writer.add_scalar('Elapsed Time/valid-epoch-secs', valid_end_time - valid_start_time, epoch)

    print(f"Saving epoch {epoch+1}...")
    torch.save(model.state_dict(), f"checkpoints/{model_name}/epoch-{epoch+1}.pth") # checkpoint per epoch for safety

    if valid_epoch_loss < best_valid_loss:
        best_valid_loss = valid_epoch_loss
        epochs_without_gain = 0

        torch.save(model.state_dict(), f"checkpoints/{model_name}/best_model.pth") # checkpoint the best model so far
    else:
        epochs_without_gain += 1

    if epochs_without_gain >= early_stopping_rounds:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("Train Acc:", max_train_acc)
print("Valid Acc:", max_valid_acc)

writer.flush()

**Display confusion matrix for valid data**

In [ ]:
cm = confusion_matrix(valid_epoch_labels, valid_epoch_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.class_names)
disp.plot()

**Show classification report**

In [ ]:
print(classification_report(valid_epoch_labels, valid_epoch_preds))